# Ground truth labeling

Enter the name of the CSV file you want to label (e.g. `ground_truth_labeling_1.csv` … `ground_truth_labeling_5.csv`), then click **Load**. Progress is saved to the CSV after each label. When you load again, the UI starts at the **first row that has no ground truth label** so you can continue where you left off.

In [3]:
import pandas as pd
import os
import re
import ipywidgets as widgets
from IPython.display import display

In [4]:
filename_text = widgets.Text(
    value="ground_truth_labeling_1.csv",
    description="CSV file:",
    layout=widgets.Layout(width="320px")
)
load_btn = widgets.Button(description="Load")

def on_load(btn):
    label_csv = filename_text.value.strip()
    if not label_csv:
        print("Enter a CSV filename.")
        return
    if not os.path.exists(label_csv):
        print(f"File not found: {label_csv}")
        return
    _df = pd.read_csv(label_csv, dtype={"label": str})
    _df["ground_truth_lon"] = pd.to_numeric(_df["ground_truth_lon"], errors="coerce")
    _df["ground_truth_lat"] = pd.to_numeric(_df["ground_truth_lat"], errors="coerce")
    n = len(_df)
    # Start from first row without a ground truth label (resume progress)
    label_str = _df["label"].astype(str).str.strip().str.lower()
    unlabeled = label_str.isna() | (label_str == "") | (label_str == "nan")
    start_idx = int(_df.index[unlabeled][0]) if unlabeled.any() else 0
    n_labeled = n - unlabeled.sum()
    print(f"Loaded {n} rows from {label_csv}. Already labeled: {n_labeled}. Starting at row {start_idx + 1} (first unlabeled).")

    idx = widgets.IntSlider(min=0, max=max(0, n - 1), step=1, value=start_idx, description="Index")
    info = widgets.HTML(value="")
    gt_lon = widgets.Text(value="", description="GT lon", placeholder="optional")
    gt_lat = widgets.Text(value="", description="GT lat", placeholder="optional")
    paste_coords = widgets.Text(value="", placeholder="Paste coords: e.g. 41.618, -88.206", layout=widgets.Layout(width="400px"))
    fill_btn = widgets.Button(description="Fill from paste")

    def update_display(index):
        if index < 0 or index >= len(_df):
            info.value = "No row at this index."
            return
        r = _df.iloc[index]
        name = r.get("name", "")
        address = r.get("address", "")
        lon, lat = r.get("lon"), r.get("lat")
        osm = r.get("map_osm", "")
        gl = r.get("map_google", "")
        lbl = r.get("label", "")
        info.value = f"""
        <b>Place {index + 1} / {len(_df)}</b><br>
        <b>Name:</b> {name}<br>
        <b>Address:</b> {address}<br>
        <b>Current pin:</b> {lon}, {lat}<br>
        <b>Your label:</b> {lbl}<br>
        <a href="{osm}" target="_blank">Open in OSM</a> &nbsp;
        <a href="{gl}" target="_blank">Open in Google Maps</a>
        """
        lo, la = r.get("ground_truth_lon"), r.get("ground_truth_lat")
        gt_lon.value = str(lo).strip() if pd.notna(lo) and str(lo).strip() else ""
        gt_lat.value = str(la).strip() if pd.notna(la) and str(la).strip() else ""

    def _parse_opt_float(s):
        s = (s or "").strip()
        if not s:
            return None
        try:
            return float(s)
        except ValueError:
            return None

    def _parse_pasted_coords(s):
        s = (s or "").strip()
        if not s:
            return None, None
        parts = re.split(r"[, \t\n]+", s)
        nums = []
        for p in parts:
            p = p.strip()
            if not p:
                continue
            try:
                nums.append(float(p))
            except ValueError:
                continue
        if len(nums) < 2:
            return None, None
        a, b = nums[0], nums[1]
        if -90 <= a <= 90 and -180 <= b <= 180:
            return b, a
        if -180 <= a <= 180 and -90 <= b <= 90:
            return a, b
        return b, a

    def on_fill_paste(btn):
        lon, lat = _parse_pasted_coords(paste_coords.value)
        if lon is not None and lat is not None:
            gt_lon.value = str(lon)
            gt_lat.value = str(lat)
            print(f"Filled GT lon={lon}, lat={lat}")
        else:
            print("Could not parse two numbers.")

    def on_label(btn):
        i = idx.value
        if i < 0 or i >= len(_df):
            return
        r = _df.iloc[i]
        _df.at[_df.index[i], "label"] = btn.description
        if btn.description == "Accurate":
            _df.at[_df.index[i], "ground_truth_lon"] = float(r["lon"])
            _df.at[_df.index[i], "ground_truth_lat"] = float(r["lat"])
        else:
            lo, la = _parse_opt_float(gt_lon.value), _parse_opt_float(gt_lat.value)
            if lo is not None and la is not None:
                _df.at[_df.index[i], "ground_truth_lon"] = lo
                _df.at[_df.index[i], "ground_truth_lat"] = la
            else:
                _df.at[_df.index[i], "ground_truth_lon"] = float("nan")
                _df.at[_df.index[i], "ground_truth_lat"] = float("nan")
        _df.to_csv(label_csv, index=False)
        idx.value = min(i + 1, len(_df) - 1)
        update_display(idx.value)
        print(f"Saved '{btn.description}' for index {i} -> {label_csv}")

    def on_idx_change(change):
        update_display(change["new"])

    fill_btn.on_click(on_fill_paste)
    idx.observe(on_idx_change, names="value")
    update_display(start_idx)

    btns = [widgets.Button(description=lbl) for lbl in ("Accurate", "Inaccurate", "Skip")]
    for b in btns:
        b.on_click(on_label)

    display(widgets.VBox([
        widgets.HBox([idx, *btns]),
        info,
        widgets.HBox([gt_lon, gt_lat]),
        widgets.HBox([paste_coords, fill_btn]),
        widgets.HTML(value="<small>Paste coords → Fill from paste → Accurate/Inaccurate/Skip to save.</small>"),
    ]))

load_btn.on_click(on_load)
display(widgets.VBox([
    widgets.HBox([filename_text, load_btn]),
    widgets.HTML(value="<small>Examples: ground_truth_labeling_1.csv … ground_truth_labeling_5.csv</small>"),
]))